In [1]:
from src.models.cvmodels import FeatureBasedEstimator
from src.models.feature_extractors import RapidTextExtractor, OrbFeatureExtractor, SIFTFeatureExtractor
from src.models.deepmodels import SiameseDino
from transformers import AutoImageProcessor, AutoModel
from hydra import initialize, compose
import torch

In [ ]:
rapidocr = RapidTextExtractor()
feature_extractors = [rapidocr]
model4 = FeatureBasedEstimator(feature_extractors, [1])

ValueError: Weights must sum to 1.0, got 0.75

In [ ]:
with initialize(config_path="../config", version_base=None):
    cfg = compose(config_name="base_config")

checkpoint = torch.load("../model_checkpoints/sandy-fire-218.pth")

backbone_name = "facebook/dinov3-vitb16-pretrain-lvd1689m"
processor = AutoImageProcessor.from_pretrained(backbone_name)
backbone = AutoModel.from_pretrained(backbone_name, dtype=torch.float32)

model1 = SiameseDino(backbone, processor, cfg)
model1.load_state_dict(checkpoint)

rapidocr = RapidTextExtractor()
orb = OrbFeatureExtractor(1000)
sift = SIFTFeatureExtractor()
feature_extractors = [orb]
weights = [1]
model2 = FeatureBasedEstimator(feature_extractors, weights)
feature_extractors = [orb, rapidocr]
weights = [0.5, 0.5]
model3 = FeatureBasedEstimator(feature_extractors, weights)
feature_extractors = [sift]
model4 = FeatureBasedEstimator(feature_extractors, [1])

In [ ]:
from src.utils import ModelReport
from src.eval import Recall

metric = Recall()
modelreport = ModelReport({"sift": model4})#"siamesedino": model1, "orb": model2, "0.5_orb+0.5_rapidocr": model3, "sift": model4})
df = modelreport.generate_report("../data/small_data", "random", metric)

[codecarbon WARNING @ 16:58:00] Multiple instances of codecarbon are allowed to run at the same time.


Found 18 total classes.
Processing Train classes...
Processing Validation classes...
Processing Test classes...

--- Data Split Summary ---
Training Loader:      16 samples from 1 classes (Augmented)
Gallery Loader:       56 samples from 18 classes (Original)
Val Query Loader:     17 samples from 17 classes (Original)
Test Query Loader:     0 samples from 0 classes (Original)
Loading dataloaders...
------------------- MODEL 0 ---------------------
🔨 Preparing gallery (computing features)...


Computing features: 100%|██████████| 55/55 [00:39<00:00,  1.41it/s]


✅ Gallery prepared: 55 images
Computing queries features...


Computing features: 100%|██████████| 18/18 [00:12<00:00,  1.47it/s]
/home/timothee/anaconda3/envs/DIHT/lib/python3.11/site-packages/scipy/sparse/_base.py:958: RuntimeWarning: divide by zero encountered in divide
  recip = np.true_divide(1., other)


55
(18, 73)
[[[0.66666667 0.         0.         ... 0.         0.         0.        ]
  [0.         0.         0.         ... 0.         0.         0.        ]
  [0.         0.         0.         ... 0.         0.         0.        ]
  ...
  [0.         0.         0.         ... 1.         0.         0.        ]
  [0.         0.         0.         ... 0.         0.         0.        ]
  [0.         0.         0.         ... 0.         0.         0.        ]]]
[[0.66666667 0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 1.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]
Elapsed time for evaluate = 51.42s
Energy consumption for evaluate = 0.000931 kWh
Carbon foo

TypeError: OCRExtractor.compute_distances() takes 2 positional arguments but 3 were given

In [ ]:
df

,evaluation time,inference time,recall@1,recall@3,recall@10,store size (mb)
siamesedino,3.170333,0.190323,0.579545,0.727273,0.840909,6576
orb,88.769850,0.700401,0.829545,0.863636,0.886364,6576
0.5_orb+0.5_rapidocr,318.283803,1.562125,0.715909,0.761364,0.806818,6576


In [2]:
img1 = "/home/timothee/Documents/programmation/DIHT/data/original_data/002/IMG_20240220_175710405.jpg"
img2 = "/home/timothee/Documents/programmation/DIHT/data/original_data/002/IMG_20240220_225113575.jpg"
f1 = sift.get_features(img1)
f2 = sift.get_features(img2)

In [3]:
sift.compute_distance(f1, f2)

0.002506265664160401